# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library, referencing all dataset entities by their `@id` fields for traceability and reproducibility.

### Dataset Source
The dataset is defined by a Croissant schema at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata; print key description fields
meta = dataset.metadata
print('Dataset name:', meta.name)
print('Description:', meta.description)
print('Identifier:', getattr(meta, 'identifier', 'N/A'))
print('Authors:', getattr(meta, 'author', 'N/A'))
print('Version:', getattr(meta, 'version', 'N/A'))


## 2. Data Overview
List all available record set `@id`s and the corresponding field `@id`s for reference. You'll use these IDs to extract and manipulate data.

You can access record sets via:

- `dataset.record_sets`: List of all record sets, each with an `@id`, name, and description
- Each record set's fields are accessible via `fields` attribute, each with its own `@id` and metadata

In [ ]:
# List all available record sets and their fields by @id
print('Available record sets and their fields:')
record_set_ids = []
for rs in dataset.record_sets:
    print(f"\nRecord set name: {rs.name}\n  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print('  Fields:')
    for f in rs.fields:
        print(f"    - {f.name}   (@id: {f.id})   Type: {getattr(f, 'data_type', 'N/A')}")

## 3. Data Extraction
Extract data from the desired record set(s) into DataFrames using the selected `@id`s. Throughout the notebook, entities (record sets, fields) are referenced only by their `@id` fields.

In [ ]:
# Choose desired record set(s) to load - using @id
# For this dataset, we will extract all record sets for demonstration
dataframes = {}
for rs_id in record_set_ids:
    print(f"\nLoading records from record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Extracted columns: {df.columns.tolist()}")
    print(f"  Sample records:\n{df.head(3)}")

# Select a record set to proceed with (if several exist). We'll use the first one found:
selected_rs_id = record_set_ids[0]
print(f"\nProceeding with record set: {selected_rs_id}")


## 4. Exploratory Data Analysis (EDA)
Apply basic preprocessing and explore the fields in the selected record set. We'll:   
- Show available numeric and categorical fields by their `@id`
- Filter records based on a chosen numeric field (e.g., age or similar)
- Normalize a numeric field
- Group by a categorical/grouping field and compute statistics (all using `@id` field references)

_Replace the field `@id`s with those present in your dataset output above if adjusting this notebook for other data!_

In [ ]:
df = dataframes[selected_rs_id]
# List all columns and attempt to infer types
print('All available fields (@id):')
print(df.columns.tolist())

# Identify possible numeric fields (float/int):
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_fields:
    print("Warning: No numeric fields found in the DataFrame. Please examine data types.")
else:
    print('Numeric fields (@id):', numeric_fields)

# For demonstration, select the first numeric field (override with a different @id if desired):
numeric_field_id = numeric_fields[0] if numeric_fields else None

if numeric_field_id:
    # Set an arbitrary threshold as demo (e.g., > median)
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize selected numeric field
    field_norm = f"{numeric_field_id}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, field_norm]].head())

    # Try to choose a group field (categorical)
    cat_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
    group_field_id = cat_fields[0] if cat_fields else None
    if group_field_id:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped filtered data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped)
    else:
        print('No suitable group/categorical field found for grouping.')
else:
    print("No numeric field to analyze.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if available, its relationship to a grouping field. _Requires matplotlib/seaborn._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20, color='skyblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
- This notebook demonstrated:
    - How to load and inspect a FAIR² Croissant dataset using `mlcroissant`, referencing all entities by their `@id`
    - How to extract and process data from record sets and fields via `@id`
    - Basic exploratory analysis (filtering, normalization, grouping)
    - Visualizing numeric data distributions and group-wise differences

Refer to the Croissant schema documentation and your dataset's specific field `@id`s to adapt these analyses. For reproducible data science, always reference entities by their `@id`s rather than only names or positions.